In [ ]:
!pip install -q gradio sentence-transformers faiss-cpu \
    transformers peft torch PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 74.3 MB/s eta 0:00:00


In [ ]:
import os, json, re
import numpy as np
import torch
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


Load index and chunks

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = "/content/drive/MyDrive/372-Project"
LORA_PATH  = f"{DRIVE_BASE}/models/lora-coach"

# Load FAISS index and chunks saved from rag_pipeline.ipynb
index  = faiss.read_index(f"{DRIVE_BASE}/models/notes.faiss")
with open(f"{DRIVE_BASE}/models/chunks.json") as f:
    chunks = json.load(f)

print(f"Loaded FAISS index: {index.ntotal} vectors")
print(f"Loaded {len(chunks)} chunks")

Mounted at /content/drive
Loaded FAISS index: 2 vectors
Loaded 2 chunks


Load embedder and reranker

In [ ]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device=device
)
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2", device=device
)
print("Embedder and reranker loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Embedder and reranker loaded.


Load model

In [ ]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base  = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(base, LORA_PATH)
model.eval()
print("Fine-tuned model loaded.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Fine-tuned model loaded.


Retrieval pipeline

In [ ]:
def retrieve_dense(query, k=10):
    q_emb = embedder.encode(
        [query], normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(q_emb, k)
    return [(int(i), float(s)) for i, s in zip(indices[0], scores[0]) if i >= 0]

def retrieve_and_rerank(query, retrieve_k=10, final_k=3):
    candidates  = retrieve_dense(query, k=retrieve_k)
    cand_chunks = [chunks[i] for i, _ in candidates]
    pairs       = [(query, c) for c in cand_chunks]
    scores      = reranker.predict(pairs)
    ranked      = sorted(zip(cand_chunks, scores), key=lambda x: x[1], reverse=True)
    return [(c, float(s)) for c, s in ranked[:final_k]]

Context window manager

In [ ]:
MAX_HISTORY_TOKENS = 800   # how much history to keep in prompt
MAX_CONTEXT_TOKENS = 1024  # total prompt budget

class ConversationManager:
    """
    Manages multi-turn conversation history with token-aware truncation.

    Tracks:
      - Full message history (never discarded — used for display)
      - Truncated history window (fitted to token budget for each prompt)
      - Retrieved sources per turn (for transparency)
    """

    def __init__(self):
        self.history = []        # list of {"role": "user"|"assistant", "content": str}
        self.sources = []        # retrieved chunks per turn

    def add_turn(self, user_msg, assistant_msg, sources):
        self.history.append({"role": "user",      "content": user_msg})
        self.history.append({"role": "assistant", "content": assistant_msg})
        self.sources.append(sources)

    def get_history_block(self):
        """Return recent history fitted to MAX_HISTORY_TOKENS budget.
        Drops oldest turns first when over budget.
        """
        lines = []
        for turn in self.history:
            role = "Student" if turn["role"] == "user" else "Coach"
            lines.append(f"{role}: {turn['content']}")

        # Trim from oldest if over token budget
        while lines:
            joined = "\n".join(lines)
            tokens = len(tokenizer.encode(joined))
            if tokens <= MAX_HISTORY_TOKENS:
                break
            lines = lines[2:]   # drop oldest user+assistant pair

        return "\n".join(lines)

    def clear(self):
        self.history = []
        self.sources = []

    def turn_count(self):
        return len(self.history) // 2

conversation = ConversationManager()
print("ConversationManager ready.")

ConversationManager ready.


Prompt builder with history

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "What is backpropagation?",
        "answer":   "Backpropagation computes gradients of the loss with respect to "
                    "each weight by applying the chain rule layer by layer, propagating "
                    "error signals from output back to input."
    },
    {
        "question": "Why do we use activation functions?",
        "answer":   "Activation functions introduce nonlinearity so the network can "
                    "learn complex patterns. Without them, stacked linear layers "
                    "collapse to a single linear transformation."
    }
]

def build_prompt(query, retrieved_chunks, history_block=""):
    """Chain-of-thought prompt with conversation history injected.
    History block is empty on the first turn.
    """
    context = "\n\n".join(
        f"[Source {i+1}]\n{c}" for i, (c, _) in enumerate(retrieved_chunks)
    )

    examples = "\n\n".join(
        f"Q: {ex['question']}\nA: {ex['answer']}"
        for ex in FEW_SHOT_EXAMPLES
    )

    history_section = ""
    if history_block:
        history_section = f"""Previous conversation:
{history_block}

"""

    return f"""You are a helpful AI learning coach. Use the notes below to answer the student's question clearly and accurately.

Examples of good answers:
{examples}

Notes:
{context}

{history_section}Let's think through this step by step:
1. Identify the key concept in the question.
2. Find the most relevant information in the notes.
3. Explain it clearly for a student learning this for the first time.

Student: {query}
Coach:"""

Generation function

In [ ]:
def generate(prompt, temperature=0.7, max_new_tokens=300):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_CONTEXT_TOKENS
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if "Coach:" in decoded:
        return decoded.split("Coach:")[-1].strip()
    return decoded

chat function

In [ ]:
def chat(user_message, gradio_history, temperature=0.7):
    """
    Called by Gradio on each user message.

    gradio_history: list of [user_msg, assistant_msg] pairs — Gradio's format
    Returns: updated gradio_history, empty textbox, sources string
    """
    if not user_message.strip():
        return gradio_history, "", "No query entered."

    # 1. Retrieve relevant chunks
    top_chunks = retrieve_and_rerank(user_message)

    # 2. Build history block from ConversationManager
    history_block = conversation.get_history_block()

    # 3. Build prompt
    prompt = build_prompt(user_message, top_chunks, history_block)

    # 4. Generate answer
    answer = generate(prompt, temperature=temperature)

    # 5. Save turn to history
    conversation.add_turn(user_message, answer, top_chunks)

    # 6. Update Gradio history (its own format for display)
    gradio_history = gradio_history + [[user_message, answer]]

    # 7. Format sources for display
    sources_text = f"**Turn {conversation.turn_count()} — Retrieved Sources:**\n\n"
    for i, (chunk, score) in enumerate(top_chunks):
        sources_text += f"**[{i+1}]** score={score:.3f}\n{chunk[:200]}...\n\n"

    return gradio_history, "", sources_text


def reset_conversation():
    """Clear history for a fresh session."""
    conversation.clear()
    return [], "", "Conversation cleared."

gradio UI

In [ ]:
with gr.Blocks(title="AI Learning Coach", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎓 AI Learning Coach
    Ask questions about your course notes. The coach remembers your conversation.
    """)

    with gr.Row():

        # ── Left column: chat ──────────────────────────────────────────────
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=500,
                show_label=True
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a question about your notes...",
                    label="Your question",
                    scale=4,
                    lines=2
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.Button("🗑 Clear Conversation", variant="secondary")

            with gr.Accordion("⚙️ Generation Settings", open=False):
                temperature_slider = gr.Slider(
                    minimum=0.1, maximum=1.5, value=0.7, step=0.05,
                    label="Temperature (higher = more creative, lower = more focused)"
                )

        # ── Right column: sources ──────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 📄 Retrieved Sources")
            gr.Markdown("Chunks retrieved from your notes that informed the answer.")
            sources_box = gr.Markdown("Sources will appear here after your first question.")

    # ── History display (for inspection / debugging) ───────────────────────
    with gr.Accordion("📋 Raw Conversation History", open=False):
        history_display = gr.JSON(label="Full history log")
        refresh_btn = gr.Button("Refresh History")

        def refresh_history():
            return conversation.history

        refresh_btn.click(refresh_history, outputs=history_display)

    # ── Event wiring ───────────────────────────────────────────────────────
    send_btn.click(
        fn=chat,
        inputs=[msg_box, chatbot, temperature_slider],
        outputs=[chatbot, msg_box, sources_box]
    )
    msg_box.submit(        # also send on Enter key
        fn=chat,
        inputs=[msg_box, chatbot, temperature_slider],
        outputs=[chatbot, msg_box, sources_box]
    )
    clear_btn.click(
        fn=reset_conversation,
        outputs=[chatbot, msg_box, sources_box]
    )

demo.launch(share=True, debug=False)  # share=True gives a public Colab URL

/tmp/ipykernel_5303/1731575782.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Learning Coach", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_5303/1731575782.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5303/1731575782.py:12: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://65a216015239106b3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://65a216015239106b3d.gradio.live
